# SfN 2026 poster figures — Tier-2 regeneration

Regenerates every poster panel in the **Tier-2 poster house style** (Source Sans, despined,
no grid) as **SVG / PDF** (no PNG), assuming all upstream pickles/trajectories already exist.

- **Nets:** A = `cowmobsweep_cow2p0x_mob0p5x` (world 45) · B = `…_wrld50` · C = `…_multiworld`.
- **Apples-to-apples:** B/C are *replayed on A's trajectories* (`run_replay_onA_{B,C}.sh`) →
  `layer_decode_on_A/` + `tuning_on_A/`; untrained = A's trajectories through a random net.
- Each panel section is **independently runnable** after the Setup + Check cells.
- Outputs land in each net's logdir **and** are staged into `scratch/sfn2026/<panel>/`.

> Run **Setup** and **Check data availability** first. Any ✗ in the check means run that
> pipeline step (train / eval_trajectory / tuning_curve / decode / dream_seed_ablation /
> the `run_replay_onA_*` SLURM jobs) before the corresponding panel cell.


## Setup

In [ ]:
import os, sys, shutil, subprocess, glob
REPO = '/gpfs/radev/project/levenstein/vv266/dreamerv3f'
os.chdir(REPO)
# both REPO and dreamerv3/ on path so top-level (`import plot_presentable`) and package
# (`from dreamerv3 import dream_seed_ablation`) imports resolve to the SAME modules.
for p in (REPO, os.path.join(REPO, 'dreamerv3')):
    if p not in sys.path: sys.path.insert(0, p)
os.environ['MPLBACKEND'] = 'Agg'

import matplotlib.pyplot as plt
import plot_presentable as pp
pp.apply_presentable_style(poster=True, small_figsize=False)   # Tier-2 for inline cells

# --- net logdirs ---
A = 'logdir/default_masked_25m_cowmobsweep_cow2p0x_mob0p5x'
B = 'logdir/default_masked_25m_cowmob_cow2p0x_mob0p5x_wrld50'
C = 'logdir/default_masked_25m_cowmob_cow2p0x_mob0p5x_multiworld'
A_ENV_SEED = 616548150          # world 45 = hash((45,0)) % (2**32-1); for manifold colorkey
OUT = 'scratch/sfn2026'
for d in ['panel2_dreamerv3','panel3','panel4','panel5','panel6','provenance']:
    os.makedirs(f'{OUT}/{d}', exist_ok=True)

PY = sys.executable
ENV = {**os.environ, 'PYTHONPATH': f'{REPO}:{REPO}/dreamerv3', 'MPLBACKEND':'Agg'}

def sh(cmd):
    '''Run a CLI command from REPO; print the tail of its output.'''
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, cwd=REPO, env=ENV,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = '\n'.join(r.stdout.strip().splitlines()[-8:])
    print(tail if tail else '(no output)')
    if r.returncode: print(f'  !! exit {r.returncode}')
    return r.returncode == 0

def stage(src, dst):
    if os.path.exists(src):
        shutil.copy(src, f'{OUT}/{dst}'); print(f'  staged -> {OUT}/{dst}')
    else:
        print(f'  MISSING {src}')

def show(path, width=560):
    '''Inline preview an svg/pdf/png.'''
    from IPython.display import Image, display
    p = path if os.path.isabs(path) else os.path.join(REPO, path)
    if not os.path.exists(p): print('  (no file to preview)', p); return
    try:
        if p.endswith('.svg'):
            import cairosvg, io; png = cairosvg.svg2png(url=p, output_width=1200)
            display(Image(data=png, width=width))
        elif p.endswith('.pdf'):
            import pypdfium2 as pdfium
            pg = pdfium.PdfDocument(p)[0].render(scale=1.3).to_pil()
            import io; b=io.BytesIO(); pg.save(b,'PNG'); display(Image(data=b.getvalue(), width=width))
        else:
            display(Image(filename=p, width=width))
    except Exception as e:
        print('  (preview unavailable:', e, ')')
print('ready. A/B/C logdirs set; Tier-2 poster style applied for inline cells.')

## Check data availability
Verifies every upstream pickle/input exists. ✗ → run that pipeline step first.

In [ ]:
REQUIRED = {
 'panel2 (A)': [f'{A}/scores.jsonl', f'{A}/metrics.jsonl', f'{A}/trajectories'],
 'panel3 (A)': [f'{A}/tuning_results/tuning_results.pkl', f'{A}/dream_results/dream_results.pkl'],
 'panel3 (A/B/untr decode)': [f'{A}/layer_decoder_results/layer_decode_results.pkl',
                              f'{B}/layer_decode_on_A/layer_decode_results.pkl',
                              f'{A}/layer_decoder_results_untrained/layer_decode_results.pkl',
                              f'{B}/tuning_on_A/tuning_results.pkl'],
 'panel4 (A)': [f'{A}/trajectories'],
 'panel5 (C)': [f'{C}/scores.jsonl', f'{C}/trajectories'],
 'panel5 (A/C/untr)': [f'{C}/layer_decode_on_A/layer_decode_results.pkl',
                       f'{C}/tuning_on_A/tuning_results.pkl',
                       f'{C}/tuning_on_A/dyn_deter/example_tuning_curves_ev.pdf'],
 'panel6 (A)': [f'{A}/dream_seed_ablation_16/dream_seed_ablation_results.pkl',
                f'{A}/manifold_results/dyn_deter_isomap_wakedream.svg  (made by panel-3 manifold cell)'],
}
ok_all = True
for panel, files in REQUIRED.items():
    print(f'\n[{panel}]')
    for f in files:
        path = f.split('  ')[0]
        exists = os.path.exists(os.path.join(REPO, path))
        ok_all &= exists
        print(f'  {"OK " if exists else "MISS"}  {f}')
print('\n==>', 'ALL PRESENT' if ok_all else 'some inputs MISSING (see above)')

## Panel 2 — DreamerV3 (net A)
Training progress (combined: achievements/ep · episode duration · unlock-by-tier · score · losses),
occupancy, example vitals-episodes, and the p(action|vital)/p(action) gain (subset).

In [ ]:
# 2a — training progress (combined multi-panel)
sh(f'{PY} dreamerv3/plot_training.py --logdir {A} --save {A}/plots --smooth 50 --presentable --out_fmt svg')
stage(f'{A}/plots/training_progress.svg', 'panel2_dreamerv3/A_training_progress.svg')
show(f'{A}/plots/training_progress.svg')

In [ ]:
# 2b — occupancy (A is fixed_layout world 45)
sh(f'{PY} dreamerv3/decode_position.py --data {A}/trajectories --save {A}/occupancy '
   f'--mode occupancy --max_episodes 0 --fixed_layout --presentable --out_fmt svg')
stage(f'{A}/occupancy/occupancy_fullset_fullset.svg', 'panel2_dreamerv3/A_occupancy.svg')
show(f'{A}/occupancy/occupancy_fullset_fullset.svg')

In [ ]:
# 2c — vitals-episodes (10; pick one) + gain subset p(action|vital)/p(action)
sh(f'{PY} dreamerv3/vital_dynamics.py --data {A}/trajectories --save {A}/vital_dynamics '
   f'--max_episode_plots 10 --presentable --fmt svg')
for f in sorted(glob.glob(f'{A}/vital_dynamics/vitals_episode_0*.svg')):
    stage(f, f'panel2_dreamerv3/{os.path.basename(f)}')
stage(f'{A}/vital_dynamics/achievement_lift_vs_vital_subset.svg', 'panel2_dreamerv3/A_gain_subset.svg')
show(f'{A}/vital_dynamics/vitals_episode_003.svg')
show(f'{A}/vital_dynamics/achievement_lift_vs_vital_subset.svg')

## Panel 3 — spatial representation (net A + B-on-A + untrained)
Manifold (sRSA, Isomap of wake position), metric-isomap coloured by coherence (EV>0.5),
deter example tuning curves (EV-sorted), and the cross-condition decode comparisons (A/B/untrained).

In [ ]:
# 3a — manifold: sRSA + Isomap wake position (+ wake-vs-dream used by panel 6)
sh(f'{PY} dreamerv3/manifold_analysis.py --data {A}/trajectories '
   f'--dream_data {A}/dream_results/dream_results.pkl --save {A}/manifold_results '
   f'--max_wake_samples 4000 --n_neighbors 150 --ev_thresh 0.5 '
   f'--tuning_pkl {A}/tuning_results/tuning_results.pkl --world_env_seed {A_ENV_SEED} '
   f'--presentable --theme light --fmt svg')
stage(f'{A}/manifold_results/dyn_deter_srsa.svg', 'panel3/A_sRSA.svg')
stage(f'{A}/manifold_results/dyn_deter_isomap_position.svg', 'panel3/A_isomap_wake_position.svg')
stage(f'{A}/manifold_results/dyn_deter_position_colorkey_world.svg', 'panel3/A_isomap_position_colorkey.svg')
show(f'{A}/manifold_results/dyn_deter_isomap_position.svg')
show(f'{A}/manifold_results/dyn_deter_srsa.svg')

In [ ]:
# 3b — metric-isomap coloured by coherence (neurons EV>0.5)
sh(f'{PY} dreamerv3/analyze_tuning.py --from_pkl {A}/tuning_results/tuning_results.pkl '
   f'--save {A}/tuning_results/cluster_plots/ev0p5 --mode metrics --layers dyn/deter '
   f'--ev_thresh 0.5 --presentable')
stage(f'{A}/tuning_results/cluster_plots/ev0p5/dyn_deter_isomap_metrics_coherence.svg',
      'panel3/A_isomap_coherence_ev0p5.svg')
show(f'{A}/tuning_results/cluster_plots/ev0p5/dyn_deter_isomap_metrics_coherence.svg')

In [ ]:
# 3c — deter example tuning curves, sorted by EV (inline re-plot from pkl, poster)
import pickle, tuning_curve as TC
pp.apply_presentable_style(poster=True, small_figsize=False)
d = pickle.load(open(f'{A}/tuning_results/tuning_results.pkl','rb')); L = d['layers']['dyn/deter']
outpdf = f'{A}/tuning_results/example_tuning_curves_deter_EVsorted_poster.pdf'
TC.plot_example_tuning_curves(L['tuning_curves'], L['metrics'], L['group_ids'], 'dyn/deter',
    outpdf, n_examples=20, area=(32,32), sort_by='EV', ev_filter=0.0)
stage(outpdf, 'panel3/A_deter_tuning_curves.pdf')
show(outpdf)

In [ ]:
# 3d — multiworld comparison boxplot: A vs B vs untrained (key layers; untrained = bold border)
sh(f'{PY} dreamerv3/compare_conditions.py '
   f'--conditions "{A}:A_same45:decode=layer_decoder_results" '
   f'"{B}:B_newworld50:decode=layer_decode_on_A" '
   f'"{A}:Untrained:decode=layer_decoder_results_untrained" '
   f'--save {B}/sfn_compare_svg --no_tuning --no_manifold --key_layers --no_sig --presentable --out_fmt svg')
stage(f'{B}/sfn_compare_svg/decode_boxplot.svg', 'panel3/multiworld_compare_A_B_untrained.svg')
show(f'{B}/sfn_compare_svg/decode_boxplot.svg')

In [ ]:
# 3e — deter decode-error distribution overlay: A vs B vs untrained
sh(f'{PY} dreamerv3/plot_decode_error_dist.py '
   f'--cond "A={A}/layer_decoder_results/layer_decode_results.pkl" '
   f'--cond "B={B}/layer_decode_on_A/layer_decode_results.pkl" '
   f'--cond "untrained={A}/layer_decoder_results_untrained/layer_decode_results.pkl" '
   f'--layer dyn/deter --save {B}/sfn_compare_svg --out_name deter_decode_error_A_B_untrained.svg --presentable')
stage(f'{B}/sfn_compare_svg/deter_decode_error_A_B_untrained.svg', 'panel3/deter_decode_error_A_B_untrained.svg')
show(f'{B}/sfn_compare_svg/deter_decode_error_A_B_untrained.svg')

In [ ]:
# 3f — neuron-EV distribution overlay: A vs B
sh(f'{PY} dreamerv3/plot_ev_dist.py '
   f'--cond "A={A}/tuning_results/tuning_results.pkl" '
   f'--cond "B={B}/tuning_on_A/tuning_results.pkl" '
   f'--layer dyn/deter --metric EVs --save {B}/sfn_compare_svg --out_name EV_dist_A_vs_B.svg --presentable')
stage(f'{B}/sfn_compare_svg/EV_dist_A_vs_B.svg', 'panel3/EV_dist_A_vs_B.svg')
show(f'{B}/sfn_compare_svg/EV_dist_A_vs_B.svg')

## Panel 4 — state/time remapping (net A)
Vital-conditioned place fields (food & drink) + remap-vs-EV, and the day/night (2-bin) analog.

In [ ]:
# 4a — vital-conditioned place fields (food + drink), per-vital
sh(f'{PY} dreamerv3/vital_place_fields.py --data {A}/trajectories --save {A}/vital_place_fields '
   f'--layer dyn/deter --vitals food,drink --presentable --fmt svg')
for v in ('food','drink'):
    stage(f'{A}/vital_place_fields/vital_place_fields_dyn_deter_{v}.svg', f'panel4/A_vital_remap_{v}.svg')
    stage(f'{A}/vital_place_fields/vital_place_fields_population_dyn_deter_{v}.svg', f'panel4/A_vital_remapVsEV_{v}.svg')
show(f'{A}/vital_place_fields/vital_place_fields_dyn_deter_food.svg')

In [ ]:
# 4b — day/night place fields (2-bin peak-window day/night)
sh(f'{PY} dreamerv3/daynight_place_fields.py --data {A}/trajectories --save {A}/daynight_place_fields '
   f'--layer dyn/deter --presentable --fmt svg')
stage(f'{A}/daynight_place_fields/daynight_place_fields_dyn_deter.svg', 'panel4/A_daylight_remap_fields.svg')
stage(f'{A}/daynight_place_fields/daynight_place_fields_population_dyn_deter.svg', 'panel4/A_daylight_remapVsEV.svg')
show(f'{A}/daynight_place_fields/daynight_place_fields_dyn_deter.svg')

## Panel 5 — multiworld net (C) + A/C/untrained comparison
C's own behaviour (training, occupancy) + the A/C/untrained cross-condition decode comparisons
(C replayed on A's trajectories) + C-on-A deter tuning curves.

In [ ]:
# 5a/5b — C training + C occupancy (C's own world-45 rollouts)
sh(f'{PY} dreamerv3/plot_training.py --logdir {C} --save {C}/plots --smooth 50 --presentable --out_fmt svg')
stage(f'{C}/plots/training_progress.svg', 'panel5/C_training_progress.svg')
sh(f'{PY} dreamerv3/decode_position.py --data {C}/trajectories --save {C}/occupancy '
   f'--mode occupancy --max_episodes 0 --fixed_layout --presentable --out_fmt svg')
stage(f'{C}/occupancy/occupancy_fullset_fullset.svg', 'panel5/C_occupancy.svg')
show(f'{C}/plots/training_progress.svg'); show(f'{C}/occupancy/occupancy_fullset_fullset.svg')

In [ ]:
# 5c — multiworld comparison boxplot: A vs C vs untrained
sh(f'{PY} dreamerv3/compare_conditions.py '
   f'--conditions "{A}:A_same45:decode=layer_decoder_results" '
   f'"{C}:C_multiworld:decode=layer_decode_on_A" '
   f'"{A}:Untrained:decode=layer_decoder_results_untrained" '
   f'--save {C}/sfn_compare_svg --no_tuning --no_manifold --key_layers --no_sig --presentable --out_fmt svg')
stage(f'{C}/sfn_compare_svg/decode_boxplot.svg', 'panel5/multiworld_compare_A_C_untrained.svg')
show(f'{C}/sfn_compare_svg/decode_boxplot.svg')

In [ ]:
# 5d — deter decode-error overlay: A vs B vs C vs untrained (4 conditions)
sh(f'{PY} dreamerv3/plot_decode_error_dist.py '
   f'--cond "A={A}/layer_decoder_results/layer_decode_results.pkl" '
   f'--cond "B={B}/layer_decode_on_A/layer_decode_results.pkl" '
   f'--cond "C={C}/layer_decode_on_A/layer_decode_results.pkl" '
   f'--cond "untrained={A}/layer_decoder_results_untrained/layer_decode_results.pkl" '
   f'--layer dyn/deter --save {C}/sfn_compare_svg --out_name deter_decode_error_A_B_C_untrained.svg --presentable')
stage(f'{C}/sfn_compare_svg/deter_decode_error_A_B_C_untrained.svg', 'panel5/deter_decode_error_A_B_C_untrained.svg')
show(f'{C}/sfn_compare_svg/deter_decode_error_A_B_C_untrained.svg')

In [ ]:
# 5e — neuron-EV distribution overlay: A vs C ; 5f — C-on-A deter tuning curves (EV-sorted)
sh(f'{PY} dreamerv3/plot_ev_dist.py '
   f'--cond "A={A}/tuning_results/tuning_results.pkl" '
   f'--cond "C={C}/tuning_on_A/tuning_results.pkl" '
   f'--layer dyn/deter --metric EVs --save {C}/sfn_compare_svg --out_name EV_dist_A_vs_C.svg --presentable')
stage(f'{C}/sfn_compare_svg/EV_dist_A_vs_C.svg', 'panel5/EV_dist_A_vs_C.svg')
stage(f'{C}/tuning_on_A/dyn_deter/example_tuning_curves_ev.pdf', 'panel5/C_on_A_deter_tuning_curves.pdf')
show(f'{C}/sfn_compare_svg/EV_dist_A_vs_C.svg'); show(f'{C}/tuning_on_A/dyn_deter/example_tuning_curves_ev.pdf')

## Panel 6 — wake vs dream (net A)
Wake-vs-dream Isomap (from the panel-3 manifold run) + seed-ablation curves (mean ± 1 std band).
*(Decoded-dream-over-real-trajectory is handled separately — not regenerated here.)*

In [ ]:
# 6a — wake vs dream Isomap (produced by the panel-3 manifold cell; re-stage)
stage(f'{A}/manifold_results/dyn_deter_isomap_wakedream.svg', 'panel6/A_wake_vs_dream_isomap.svg')
show(f'{A}/manifold_results/dyn_deter_isomap_wakedream.svg')

In [ ]:
# 6b — seed-ablation curves (mean ± 1 std band), inline replot from pkl (poster + light)
from dreamerv3 import dream_seed_ablation as DSA
pp.apply_presentable_style(poster=True, small_figsize=False)
pp._install_format_override('svg')     # force svg output
DSA.THEME = 'light'
DSA.replot_from_pkl(f'{A}/dream_seed_ablation_16/dream_seed_ablation_results.pkl',
                    f'{A}/dream_seed_ablation_16/poster_band',
                    central='mean', shading='band', make_gif=False, real_style='ghost')
stage(f'{A}/dream_seed_ablation_16/poster_band/seed_ablation_curves.svg', 'panel6/A_seed_ablation_curves.svg')
show(f'{A}/dream_seed_ablation_16/poster_band/seed_ablation_curves.svg', width=900)

## Done
All staged under `scratch/sfn2026/panel{2..6}/`. Panel 1 (spherical cow) is an Illustrator schematic.
Note: `seed_ablation_curves` has font overlaps to tidy in Illustrator; the multiworld boxplots mark
**untrained with a bold black border** and A/B/C by fill opacity.